# Notebook 3: Multimodal Fusion and Classification

**Project:** Saraswati AI Music Technical Quality Detector
**Author:** Ankit Sharda · MSc Data Science and AI · University of Hull

---

## What this notebook does

Merges the audio features from Notebook 1 (`outputs/audio_features.csv`) and the lyric
features from Notebook 2 (`outputs/lyric_features.csv`), trains five classifiers across
five feature conditions (audio only, lyrics handcrafted, lyrics embeddings, lyrics
combined, fused), and produces the results reported in the dissertation.


# 0. Install dependencies


In [ ]:

%pip install -q scikit-learn xgboost shap pandas numpy matplotlib seaborn scipy librosa joblib

## 1. Imports and configuration

In [ ]:
# Imports, Paths and Settings

import warnings
warnings.filterwarnings('ignore',category=FutureWarning)

import copy,json,time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy import stats

from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report,confusion_matrix,accuracy_score,f1_score,
    roc_auc_score,precision_score,recall_score,roc_curve,
    average_precision_score,precision_recall_curve
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

OUTPUT_DIR=Path('outputs')
AUDIO_CSV=OUTPUT_DIR/'audio_features.csv'
LYRICS_CSV=OUTPUT_DIR/'lyrics_features.csv'
LYRICS_EMB_NPY=OUTPUT_DIR/'lyrics_embeddings.npy'
LYRICS_EMB_META=OUTPUT_DIR/'lyrics_embedding_metadata.csv'
MUQ_CSV=OUTPUT_DIR/'muq_eval_scores.csv'
RESULTS_DIR=OUTPUT_DIR/'fusion_results'

OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

RANDOM_SEED=42  # fixed for reproducibility: train/val/test split, GridSearchCV folds, MLP weight init, permutation importance
CV_FOLDS=5
TRAIN_RATIO=0.60
VAL_RATIO=0.20
TEST_RATIO=0.20
np.random.seed(RANDOM_SEED)

print('Config ready. Split: %.0f%% / %.0f%% / %.0f%%'%(
    TRAIN_RATIO*100,VAL_RATIO*100,TEST_RATIO*100
))

files={
    'Audio features':AUDIO_CSV,
    'Lyrics features':LYRICS_CSV,
    'Lyrics embeddings':LYRICS_EMB_NPY,
    'Embedding metadata':LYRICS_EMB_META,
    'MuQ scores':MUQ_CSV
}
for name,path in files.items():
    print(f'{name}:', 'FOUND' if path.exists() else 'MISSING')

## 2. Figure house style

In [ ]:
import matplotlib as mpl

# Colour-blind safe palette. These colours are distinguishable
# for the most common forms of colour-vision deficiency.
PALETTE={
    'ai':'#D55E00',
    'human':'#0072B2',
    'accent':'#009E73',
    'accent2':'#E69F00',
    'neutral':'#666666',
    'light':'#CCCCCC'
}
SERIES_COLORS=['#0072B2','#E69F00','#009E73','#CC79A7','#56B4E9']
CLASS_COLORS={'AI':PALETTE['ai'],'Human':PALETTE['human']}

mpl.rcParams.update({
    'figure.dpi':110,
    'savefig.dpi':300,
    'savefig.bbox':'tight',
    'font.family':'sans-serif',
    'font.sans-serif':['DejaVu Sans','Arial','Helvetica'],
    'font.size':10,
    'axes.titlesize':11,
    'axes.titleweight':'semibold',
    'axes.labelsize':10,
    'axes.spines.top':False,
    'axes.spines.right':False,
    'axes.grid':True,
    'axes.axisbelow':True,
    'grid.color':'#DDDDDD',
    'grid.linewidth':0.7,
    'legend.frameon':False,
    'legend.fontsize':9,
    'xtick.labelsize':9,
    'ytick.labelsize':9,
    'xtick.direction':'out',
    'ytick.direction':'out'
})

def save_fig(fig,name,outdir=RESULTS_DIR):
    outdir=Path(outdir)
    outdir.mkdir(parents=True,exist_ok=True)
    fig.savefig(outdir/f'{name}.png')
    fig.savefig(outdir/f'{name}.pdf')
    print(f'Saved: {outdir}/{name}.png and .pdf')

def tidy(ax,ygrid=True):
    ax.grid(axis='y' if ygrid else 'x',alpha=0.6)
    ax.grid(axis='x' if ygrid else 'y',visible=False)
    return ax

print('House style applied.')

## 3. Load and merge features

Audio features (NB1) and lyric features (NB2) are merged on `song_id`. Instrumental tracks without transcripts are kept for audio-only experiments but excluded from fused experiments.

In [ ]:
# Load and Merge Audio, Lyrics and Lyric Embeddings
# Features are merged here but NOT filtered yet. VarianceThreshold runs
# inside each model pipeline at fit time, so only training-set variance
# is used to decide which columns to drop. Filtering here would use
# test-set information and could cause subtle data leakage.

def true_mask(series):
    return series.astype(str).str.strip().str.lower().isin(['true','1','yes'])  # normalise boolean-like strings

required=[AUDIO_CSV,LYRICS_CSV,LYRICS_EMB_NPY,LYRICS_EMB_META]
missing=[str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing files:\n'+'\n'.join(missing))

audio_df=pd.read_csv(AUDIO_CSV,low_memory=False)
audio_df['song_id']=audio_df['song_id'].astype(str).str.strip()
if 'ok' in audio_df.columns:
    audio_df=audio_df.loc[true_mask(audio_df['ok'])].copy()
audio_df=audio_df.drop_duplicates('song_id',keep='last')
audio_df=audio_df[audio_df['source'].isin(['AI','Human'])].reset_index(drop=True)

lyrics_df=pd.read_csv(LYRICS_CSV,low_memory=False)
lyrics_df['song_id']=lyrics_df['song_id'].astype(str).str.strip()
if 'transcript_ok' in lyrics_df.columns:
    lyrics_df=lyrics_df.loc[true_mask(lyrics_df['transcript_ok'])].copy()
lyrics_df=lyrics_df.drop_duplicates('song_id',keep='last').reset_index(drop=True)

emb_matrix=np.load(LYRICS_EMB_NPY,allow_pickle=False)
emb_meta=pd.read_csv(LYRICS_EMB_META,low_memory=False)
if emb_matrix.ndim!=2:
    raise ValueError(f'Expected 2D embedding matrix, found {emb_matrix.shape}')
if len(emb_matrix)!=len(emb_meta):
    raise ValueError(f'Embedding mismatch: matrix={len(emb_matrix)}, metadata={len(emb_meta)}')
if 'song_id' not in emb_meta.columns:
    raise KeyError('song_id missing from lyrics embedding metadata.')

emb_cols=[f'emb_{i:03d}' for i in range(emb_matrix.shape[1])]
emb_df=pd.DataFrame(emb_matrix,columns=emb_cols)
emb_df.insert(0,'song_id',emb_meta['song_id'].astype(str).str.strip().values)
emb_df=emb_df.drop_duplicates('song_id',keep='last').reset_index(drop=True)

META={
    'song_id','title','artist','language','final_genre','source',
    'audio_path','ok','error','is_mono','is_clipped',
    'transcript_ok','transcript_error','full_text','clean_text',
    'transcript_path','vocals_path','demucs_used',
    'whisper_lang','whisper_detected_lang','detected_language',
    'duration_seconds','duration_sec','label','target'
}

AUDIO_EXCLUDE={
    'lufs_integrated','rms_db','sample_peak_dbfs','true_peak_dbfs',
    'hard_clip_count','clip_rate_per_sec','is_clipped','silence_ratio',
    'pentatonic_energy_ratio','pitch_dev_mean_cents',
    'pitch_dev_std_cents','pitch_microtonal_ratio',
    'tuning_offset_cents','beat_regularity_cv',
    'mfcc_1_mean','mfcc_1_std'
}

LYRICS_EXCLUDE={
    'whisper_lang_match','whisper_avg_logprob',
    'whisper_no_speech_ratio','avg_logprob','no_speech_ratio',
    'total_lines','non_empty_lines','segment_count',
    'empty_line_ratio','avg_words_per_line','max_words_per_line'
}

def numeric_features(df,exclude):
    features=[]
    for col in df.columns:
        if col in META or col in exclude or col.startswith('muq_'):
            continue
        values=pd.to_numeric(df[col],errors='coerce')
        if values.notna().sum()==0 or values.isna().mean()>=0.50:
            continue
        df[col]=values
        features.append(col)
    return features

AUDIO_FEATS=numeric_features(audio_df,AUDIO_EXCLUDE)
LYRICS_FEATS=numeric_features(lyrics_df,LYRICS_EXCLUDE)

overlap=set(AUDIO_FEATS)&set(LYRICS_FEATS)
if overlap:
    raise ValueError(f'Audio/lyrics feature-name overlap: {sorted(overlap)}')

audio_meta=[
    c for c in ['song_id','title','artist','source','language',
                'final_genre','audio_path']
    if c in audio_df.columns
]

merged=(
    audio_df[audio_meta+AUDIO_FEATS]
    .merge(lyrics_df[['song_id']+LYRICS_FEATS],on='song_id',how='inner')
    .merge(emb_df,on='song_id',how='inner')
    .drop_duplicates('song_id',keep='last')
    .reset_index(drop=True)
)

EMBED_FEATS=emb_cols
LYRICS_COMBINED_FEATS=LYRICS_FEATS+EMBED_FEATS
FUSED_FEATS=AUDIO_FEATS+LYRICS_COMBINED_FEATS

le=LabelEncoder()
y=le.fit_transform(merged['source'].astype(str))

print('Audio successful:',len(audio_df))
print('Lyrics successful:',len(lyrics_df))
print('Embeddings:',len(emb_df),'x',len(EMBED_FEATS))
print('Matched model tracks:',len(merged))
print('Audio features:',len(AUDIO_FEATS))
print('Handcrafted lyric features:',len(LYRICS_FEATS))
print('Embedding features:',len(EMBED_FEATS))
print('Fused features:',len(FUSED_FEATS))
print('Classes:',dict(zip(le.classes_,le.transform(le.classes_))))
print('\nMatched tracks by language and source:')
print(merged.groupby(['language','source']).size().rename('n').to_string())

## 4. Lyric Repetition Audit

### 4.1 Repetition differences across languages

Lyric repetition was examined separately for English, Hindi and Mandarin tracks.
This was necessary because language structure can affect how repeated words and
phrases are measured.

This analysis is treated as a descriptive lyric audit. Language interaction variables
were not manually added to the final classifier. Instead, model performance is also
evaluated separately for each language.

In [ ]:
# effect = (2*U)/(n_AI*n_Human) - 1 gives rank-biserial correlation.
# +1 means every AI track has a higher value; -1 means every human track does.
# This is reported alongside the p-value because p-values alone don't indicate
# how large a difference is.

print('=== longest_repeat_ratio by language ===')
rows=[]
for lang in ['English','Hindi','Chinese']:
    subset=merged[merged['language']==lang]
    ai=subset[subset['source']=='AI']['longest_repeat_ratio'].dropna()
    hu=subset[subset['source']=='Human']['longest_repeat_ratio'].dropna()
    if len(ai)>5 and len(hu)>5:
        u,p=stats.mannwhitneyu(ai,hu,alternative='two-sided')
        effect=(2*u/(len(ai)*len(hu)))-1
        rows.append({
            'language':lang,
            'AI_mean':round(ai.mean(),4),
            'Human_mean':round(hu.mean(),4),
            'effect':round(effect,4),
            'p_value':p,
            'n_AI':len(ai),
            'n_Human':len(hu)
        })
        print('%-8s AI=%.4f Human=%.4f p=%.4f effect=%.3f n=(%d,%d)'%(
            lang,ai.mean(),hu.mean(),p,effect,len(ai),len(hu)
        ))
    else:
        print('%-8s insufficient data'%lang)

repeat_lang_df=pd.DataFrame(rows)
display(repeat_lang_df)

plot_df=merged[
    merged['longest_repeat_ratio'].notna()
][['language','source','longest_repeat_ratio']].copy()

fig,ax=plt.subplots(figsize=(8,4.5))
sns.boxplot(
    data=plot_df,x='language',y='longest_repeat_ratio',hue='source',
    palette=CLASS_COLORS,ax=ax,linewidth=0.8
)
ax.set_title('Longest Repeated-Span Ratio by Language and Source',fontweight='bold')
ax.set_xlabel('Language')
ax.set_ylabel('Longest repeated-span ratio')
tidy(ax)
plt.tight_layout()
save_fig(fig,'vis6_longest_repeat_ratio_by_language')
plt.show()

### 4.2 Within-Hindi transcription reliability note

The `whisper_*` diagnostic features (language match rate, average log-probability,
no-speech ratio) are excluded from the fusion model because they reflect audio acquisition
quality rather than lyrical content.

Within the Hindi subset, Whisper language-match rates differed between AI and human tracks:
AI tracks had a higher proportion of transcripts detected as Hindi, while many human tracks
were detected as Urdu or returned low log-probability scores. This is reported as a
transcription reliability observation only, because Whisper confidence depends on recording
quality and encoding as well as language.


In [ ]:
lyr_raw=pd.read_csv(LYRICS_CSV,low_memory=False)
if 'transcript_ok' in lyr_raw.columns:
    ok=lyr_raw['transcript_ok'].astype(str).str.lower().eq('true')
    lyr_raw=lyr_raw.loc[ok].copy()
lyr_raw=lyr_raw.drop_duplicates('song_id',keep='last')
# Whisper diagnostic columns (log-probability, no-speech ratio, language match)
# are removed from the feature set before modelling. They reflect recording
# quality and ASR confidence rather than lyrical content, so keeping them
# would teach the classifier to distinguish audio acquisition pipelines.
diagnostic_cols=['whisper_avg_logprob','whisper_no_speech_ratio','whisper_lang_match']
diagnostic_cols=[c for c in diagnostic_cols if c in lyr_raw.columns]
rows=[]
for lang in ['English','Hindi','Chinese']:
    subset=lyr_raw[lyr_raw['language']==lang]
    for feat in diagnostic_cols:
        ai=pd.to_numeric(subset.loc[subset['source']=='AI',feat],errors='coerce').dropna()
        hu=pd.to_numeric(subset.loc[subset['source']=='Human',feat],errors='coerce').dropna()
        if len(ai)>5 and len(hu)>5:
            u,p=stats.mannwhitneyu(ai,hu,alternative='two-sided')
            effect=(2*u/(len(ai)*len(hu)))-1
            rows.append({'language':lang,'feature':feat,'AI_median':ai.median(),'Human_median':hu.median(),'effect_size':effect,'p_value':p,'n_AI':len(ai),'n_Human':len(hu)})
whisper_audit=pd.DataFrame(rows)
if whisper_audit.empty:
    print('Whisper diagnostic features were not available.')
else:
    whisper_audit['p_adjusted']=np.minimum(whisper_audit['p_value']*len(whisper_audit),1)
    display(whisper_audit.round(4))

## 5. Train / Validation / Test split

Stratified 60/20/20. The test set is never seen during training or hyperparameter search.

In [ ]:
# Stratify jointly on source, language and genre so every combination is
# represented in train, validation and test in proportion. A split stratified
# on source alone could place all tracks of one language/genre into test.
indices=np.arange(len(merged))
strata=merged[['source','language','final_genre']].astype(str).agg('|'.join,axis=1)
train_val_idx,test_idx=train_test_split(indices,test_size=TEST_RATIO,stratify=strata,random_state=RANDOM_SEED)
val_fraction=VAL_RATIO/(TRAIN_RATIO+VAL_RATIO)  # fraction of train+val that becomes validation
train_idx,val_idx=train_test_split(train_val_idx,test_size=val_fraction,stratify=strata.iloc[train_val_idx],random_state=RANDOM_SEED)
def make_split(feature_cols):
    X=merged[feature_cols].apply(pd.to_numeric,errors='coerce')
    return X.iloc[train_idx].copy(),X.iloc[val_idx].copy(),X.iloc[test_idx].copy(),y[train_idx],y[val_idx],y[test_idx]
Xtr,Xv,Xte,ytr,yv,yte=make_split(AUDIO_FEATS)
assert not set(train_idx)&set(val_idx)
assert not set(train_idx)&set(test_idx)
assert not set(val_idx)&set(test_idx)
assert len(train_idx)+len(val_idx)+len(test_idx)==len(merged)
print('Split: Train %d (%.1f%%) | Val %d (%.1f%%) | Test %d (%.1f%%)'%(len(train_idx),100*len(train_idx)/len(merged),len(val_idx),100*len(val_idx)/len(merged),len(test_idx),100*len(test_idx)/len(merged)))
for name,idx in [('Train',train_idx),('Validation',val_idx),('Test',test_idx)]:
    print('\n'+name)
    print(merged.iloc[idx].groupby(['language','source']).size().rename('n').to_string())

## 6. Preprocessing pipeline & base models

Every model: median imputer - standard scaler - classifier.

In [ ]:
from sklearn.base import clone
def make_pipe(clf,scale=True):
    steps=[('imputer',SimpleImputer(strategy='median')),('variance',VarianceThreshold(threshold=0.0))]  # median handles skewed/outlier features; threshold=0 drops zero-variance columns
    if scale:
        steps.append(('scaler',StandardScaler()))
    steps.append(('clf',clf))
    return Pipeline(steps)

# macro-F1 is used as the GridSearchCV scoring criterion rather than accuracy
# because the matched corpus is close to but not exactly balanced. Macro-F1
# weights each class equally regardless of support.
BASE_MODELS={
    'Logistic Regression':make_pipe(LogisticRegression(max_iter=2000,random_state=RANDOM_SEED,class_weight='balanced')),
    'SVM (RBF)':make_pipe(SVC(probability=True,kernel='rbf',random_state=RANDOM_SEED,class_weight='balanced')),  # probability=True needed for AUROC
    'Random Forest':make_pipe(RandomForestClassifier(n_estimators=200,random_state=RANDOM_SEED,class_weight='balanced',n_jobs=1),scale=False),
    'XGBoost':make_pipe(XGBClassifier(objective='binary:logistic',n_estimators=200,learning_rate=0.1,max_depth=4,random_state=RANDOM_SEED,eval_metric='logloss',n_jobs=1),scale=False),
    'MLP':make_pipe(MLPClassifier(hidden_layer_sizes=(128,64,32),activation='relu',solver='adam',max_iter=500,early_stopping=True,validation_fraction=0.1,random_state=RANDOM_SEED))
}
PARAM_GRIDS={
    'Logistic Regression':{'clf__C':[0.01,0.1,1.0,10.0],'clf__penalty':['l2']},
    'SVM (RBF)':{'clf__C':[0.1,1.0,10.0],'clf__gamma':['scale','auto']},
    'Random Forest':{'clf__n_estimators':[100,200],'clf__max_depth':[None,10,20],'clf__min_samples_split':[2,5]},
    'XGBoost':{'clf__learning_rate':[0.05,0.1,0.2],'clf__max_depth':[3,4,6],'clf__n_estimators':[100,200]},
    'MLP':{'clf__hidden_layer_sizes':[(64,32),(128,64,32),(256,128,64)],'clf__alpha':[1e-4,1e-3,1e-2]}
}
def tune_model(name,base_pipeline,X_train,y_train):
    model=clone(base_pipeline)
    grid=PARAM_GRIDS.get(name,{})
    if not grid:
        model.fit(X_train,y_train)
        return model
    cv=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=RANDOM_SEED)
    search=GridSearchCV(model,grid,scoring='f1_macro',cv=cv,n_jobs=-1,refit=True,verbose=0,error_score='raise')
    search.fit(X_train,y_train)
    print('%s best: %s (CV F1=%.4f)'%(name,search.best_params_,search.best_score_))
    return search.best_estimator_
print('Models defined:',list(BASE_MODELS))

## 7. Full evaluation loop

For each feature set and model: tune on train → evaluate on val and test.

In [ ]:
# AUROC requires probability scores. Classifiers that expose predict_proba
# (LR, RF, XGBoost, MLP) use that directly. SVM uses decision_function
# (signed distance from the hyperplane), which is monotonically related
# to the predicted probability and gives valid AUROC values.
def eval_split(fitted,X,y_true):
    y_pred=fitted.predict(X)
    try:
        score=fitted.predict_proba(X)[:,1] if hasattr(fitted,'predict_proba') else fitted.decision_function(X)
        auc=roc_auc_score(y_true,score)  # works with both probabilities and decision scores
    except Exception:
        auc=np.nan
    return {'accuracy':accuracy_score(y_true,y_pred),'precision':precision_score(y_true,y_pred,average='macro',zero_division=0),'recall':recall_score(y_true,y_pred,average='macro',zero_division=0),'f1_macro':f1_score(y_true,y_pred,average='macro',zero_division=0),'auroc':auc}

FEATURE_SETS=[('audio_only',AUDIO_FEATS),('lyrics_handcrafted',LYRICS_FEATS),('lyrics_embeddings',EMBED_FEATS),('lyrics_combined',LYRICS_COMBINED_FEATS),('fused',FUSED_FEATS)]

all_results=[]

tuned_models={}

for feat_label,feat_cols in FEATURE_SETS:
    print('\n=== %s (%d features) ==='%(feat_label,len(feat_cols)))
    X_train,X_val,X_test,y_train,y_val,y_test=make_split(feat_cols)
    for name,base_pipe in BASE_MODELS.items():
        print('Tuning %s...'%name,flush=True)
        start=time.time()
        fitted=tune_model(name,base_pipe,X_train,y_train)
        training_minutes=(time.time()-start)/60
        for X_split,y_split,split_name in [(X_val,y_val,'validation'),(X_test,y_test,'test')]:
            metrics=eval_split(fitted,X_split,y_split)
            all_results.append({'feature_set':feat_label,'model':name,'split':split_name,'n_features_input':len(feat_cols),'n_samples':len(y_split),'training_minutes':training_minutes,**metrics})
        tuned_models[(feat_label,name)]={'fitted':fitted,'X_train':X_train,'X_val':X_val,'X_test':X_test,'y_train':y_train,'y_val':y_val,'y_test':y_test,'feat_cols':feat_cols}

results_df=pd.DataFrame(all_results)
metric_cols=['accuracy','precision','recall','f1_macro','auroc','training_minutes']
results_df[metric_cols]=results_df[metric_cols].round(4)
results_path=RESULTS_DIR/'ablation_results.csv'
results_df.to_csv(results_path,index=False,encoding='utf-8-sig')

print('\n=== Test Results ===')
display(results_df[results_df['split']=='test'].sort_values(['feature_set','f1_macro'],ascending=[True,False]).reset_index(drop=True))
print('Saved:',results_path)

## 8. Results tables

### 8.1 Validation set

In [ ]:
val_df=results_df[results_df['split']=='validation'].copy()
display(val_df[['feature_set','model','accuracy','precision','recall','f1_macro','auroc']].sort_values(['feature_set','f1_macro'],ascending=[True,False]).reset_index(drop=True))

### 8.2 Test set final held-out evaluation

In [ ]:
test_df=results_df[results_df['split']=='test'].copy()
display(test_df[['feature_set','model','accuracy','precision','recall','f1_macro','auroc']].sort_values(['feature_set','f1_macro'],ascending=[True,False]).reset_index(drop=True))
val_df=results_df[results_df['split']=='validation'].copy()
# idxmax() returns the first row when two models share the highest validation F1.
# In the audio-only condition LR and SVM tied; LR was selected because it
# appeared first in the dataframe (alphabetical model order).
selected=val_df.loc[val_df.groupby('feature_set')['f1_macro'].idxmax(),['feature_set','model','f1_macro']].rename(columns={'f1_macro':'validation_f1'})
selected=selected.merge(test_df[['feature_set','model','accuracy','precision','recall','f1_macro','auroc']],on=['feature_set','model'],how='left')
selected=selected.rename(columns={'f1_macro':'test_f1'}).sort_values('validation_f1',ascending=False).reset_index(drop=True)
print('\n=== Models selected using validation performance ===')
display(selected.round(4))
final_row=selected.iloc[0]
print('\nFinal selected model: %s | %s | Validation F1=%.4f | Test F1=%.4f | Test AUROC=%.4f'%(final_row['feature_set'],final_row['model'],final_row['validation_f1'],final_row['test_f1'],final_row['auroc']))

### 8.3 Ablation - does fusion help?

In [ ]:
# Pivot to a model x feature-set matrix to make it easy to compare
# whether adding lyrics to audio improves or hurts each classifier.
ablation=test_df.pivot_table(index='model',columns='feature_set',values='f1_macro').round(4)
print('=== Full ablation results ===')
display(ablation)

main_cols=[c for c in ['audio_only','lyrics_combined','fused'] if c in ablation.columns]
main_ablation=ablation[main_cols].copy()
main_ablation=main_ablation.rename(columns={'lyrics_combined':'lyrics_only'})

if {'audio_only','lyrics_only','fused'}.issubset(main_ablation.columns):
    main_ablation['best_unimodal']=main_ablation[['audio_only','lyrics_only']].max(axis=1)
    main_ablation['fusion_gain']=(main_ablation['fused']-main_ablation['best_unimodal']).round(4)
    main_ablation['fusion_gain_pp']=(main_ablation['fusion_gain']*100).round(2)

print('\n=== Main modality comparison ===')
display(main_ablation.sort_values('fused',ascending=False))

## 9. Visualisations

### VIS 7 Ablation bar chart

In [ ]:
pivot=test_df.pivot_table(index='model',columns='feature_set',values='f1_macro')
pivot=pivot.rename(columns={'lyrics_combined':'lyrics_only'})

feat_order=[c for c in ['audio_only','lyrics_only','fused'] if c in pivot.columns]
pivot=pivot[feat_order].sort_values('fused')

SET_STYLE={'audio_only':dict(color=PALETTE['human'],marker='o',label='Audio only'),'lyrics_only':dict(color=PALETTE['accent2'],marker='s',label='Lyrics only'),'fused':dict(color=PALETTE['accent'],marker='D',label='Fused')}

fig,(ax,ax2)=plt.subplots(1,2,figsize=(12,0.62*len(pivot)+3.2),gridspec_kw={'width_ratios':[3,1.15],'wspace':0.08},sharey=True)
y=np.arange(len(pivot))

for yi,(_,row) in enumerate(pivot.iterrows()):
    vals=[row[c] for c in feat_order if pd.notna(row[c])]
    ax.plot([min(vals),max(vals)],[yi,yi],color=PALETTE['light'],lw=3,solid_capstyle='round',zorder=1)
for col in feat_order:
    st=SET_STYLE[col]
    ax.scatter(pivot[col],y,s=95,zorder=3,edgecolor='white',linewidth=1.1,color=st['color'],marker=st['marker'],label=st['label'])
    for yi,val in enumerate(pivot[col]):
        ax.annotate('%.3f'%val,(val,yi),xytext=(0,9),textcoords='offset points',ha='center',va='bottom',fontsize=7.5)

ax.set_yticks(y)
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Macro-F1 on held-out test set')
ax.set_xlim(max(0.60,pivot.min().min()-0.04),min(1.01,pivot.max().max()+0.025))
ax.set_title('Performance by Feature Modality',fontweight='bold',loc='left')
ax.legend(loc='lower left',frameon=False,ncol=3,bbox_to_anchor=(0,1.02))
ax.grid(axis='x',alpha=0.55)
ax.grid(axis='y',visible=False)

for spine in ['top','right','left']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='y',length=0)

if {'audio_only','lyrics_only','fused'}.issubset(pivot.columns):
    best_unimodal=pivot[['audio_only','lyrics_only']].max(axis=1)
    delta=pivot['fused']-best_unimodal
    colors=[PALETTE['accent'] if d>=0 else PALETTE['ai'] for d in delta]
    ax2.barh(y,delta,color=colors,height=0.45,edgecolor='white',linewidth=0.7)
    ax2.axvline(0,color='black',lw=1)
    limit=max(0.01,np.abs(delta).max()*1.8)
    ax2.set_xlim(-limit,limit)
    for yi,d in enumerate(delta):
        ax2.annotate('%+.3f'%d,(d,yi),xytext=(6 if d>=0 else -6,0),textcoords='offset points',ha='left' if d>=0 else 'right',va='center',fontsize=8)
    ax2.set_xlabel('Fused − best unimodal F1')
    ax2.set_title('Fusion gain',fontweight='bold',loc='left')
    ax2.grid(axis='x',alpha=0.55)
    ax2.grid(axis='y',visible=False)
    for spine in ['top','right','left']:
        ax2.spines[spine].set_visible(False)
    ax2.tick_params(axis='y',length=0)

fig.subplots_adjust(left=0.16,right=0.97,top=0.84,bottom=0.15)
save_fig(fig,'vis7_main_modality_ablation')
plt.show()

### VIS 8 Confusion matrix (best model)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

_val_fused=results_df[(results_df['split']=='validation')&(results_df['feature_set']=='fused')]  # use validation to pick model; test set not touched here
best_row=_val_fused.loc[_val_fused['f1_macro'].idxmax()]

best_key=(best_row['feature_set'],best_row['model'])
bundle=tuned_models[best_key]
y_pred=bundle['fitted'].predict(bundle['X_test'])
y_true=bundle['y_test']

cm=confusion_matrix(y_true,y_pred)
cm_pct=cm/cm.sum(axis=1,keepdims=True)
CMAP=LinearSegmentedColormap.from_list('hull',['#FFFFFF',PALETTE['human']])

fig,ax=plt.subplots(figsize=(5.6,4.8))
ax.imshow(cm_pct,cmap=CMAP,vmin=0,vmax=1)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        val,pct=cm[i,j],cm_pct[i,j]
        col='white' if pct>0.55 else '#222222'
        ax.text(j,i-0.06,'%d'%val,ha='center',va='center',
                fontsize=19,fontweight='bold',color=col)
        ax.text(j,i+0.24,'%.1f%%'%(100*pct),ha='center',va='center',
                fontsize=10,color=col,alpha=0.85)
        if i!=j and val>0:
            ax.add_patch(plt.Rectangle((j-0.5,i-0.5),1,1,fill=False,
                                       edgecolor=PALETTE['ai'],lw=2.2,zorder=5))

ax.set_xticks(range(len(le.classes_)));ax.set_xticklabels(le.classes_,fontsize=11)
ax.set_yticks(range(len(le.classes_)));ax.set_yticklabels(le.classes_,fontsize=11)
ax.set_xlabel('Predicted',fontsize=11);ax.set_ylabel('True',fontsize=11)
ax.set_xticks(np.arange(-0.5,len(le.classes_),1),minor=True)
ax.set_yticks(np.arange(-0.5,len(le.classes_),1),minor=True)
ax.grid(which='minor',color='white',linewidth=2.5)
ax.grid(which='major',visible=False)
ax.tick_params(which='minor',length=0);ax.tick_params(length=0)

for s in ax.spines.values():
    s.set_visible(False)

acc=(y_true==y_pred).mean()
ax.set_title('Confusion matrix: %s, fused\nn=%d, accuracy=%.3f, errors=%d'%(
    best_row['model'],len(y_true),acc,(y_true!=y_pred).sum()),
    fontsize=10.5,fontweight='bold',pad=12)

plt.tight_layout()
save_fig(fig,'vis8_confusion_matrix')
plt.show()

print('Selected using validation F1: %.4f'%best_row['f1_macro'])
print(classification_report(y_true,y_pred,target_names=le.classes_,digits=4))

### VIS 9 SHAP beeswarm (XGBoost)

In [ ]:
# SHAP TreeExplainer is exact for tree-based models (XGBoost, Random Forest).
# It decomposes each prediction into per-feature contributions that sum to
# the model output. KernelExplainer would work for SVM but is much slower
# and approximate, so permutation importance is used for SVM instead (cell below).
def get_xgb_df(bundle):
    pipe=bundle['fitted']
    X=pipe.named_steps['imputer'].transform(bundle['X_test'])
    names=np.array(bundle['feat_cols'])
    if 'variance' in pipe.named_steps:  # align feature names after VarianceThreshold dropped columns
        selector=pipe.named_steps['variance']
        X=selector.transform(X)
        names=names[selector.get_support()]
    return pd.DataFrame(X,columns=names)

POS_CLASS=le.classes_[1]
NEG_CLASS=le.classes_[0]

print('Positive SHAP -> %s | Negative SHAP -> %s'%(POS_CLASS,NEG_CLASS))

for shap_key,shap_label in [
    (('audio_only','XGBoost'),'Audio-only'),
    (('fused','XGBoost'),'Fused')
]:
    bundle=tuned_models.get(shap_key)
    if bundle is None:
        continue
    X_shap=get_xgb_df(bundle)
    model=bundle['fitted'].named_steps['clf']
    explainer=shap.TreeExplainer(model)
    shap_exp=explainer(X_shap)
    sv=np.asarray(shap_exp.values)
    shap.plots.beeswarm(shap_exp,max_display=20,show=False)
    fig=plt.gcf()
    plt.title('SHAP Feature Attribution: %s XGBoost\nRight - %s | Left - %s'%(shap_label,POS_CLASS,NEG_CLASS),fontsize=10.5,fontweight='bold')
    plt.tight_layout()
    save_fig(fig,'vis9_shap_%s'%shap_key[0])
    plt.show()
    rank=pd.DataFrame({
        'feature':X_shap.columns,
        'mean_abs_shap':np.abs(sv).mean(axis=0),
        'mean_signed_shap':sv.mean(axis=0)
    }).sort_values('mean_abs_shap',ascending=False).reset_index(drop=True)
    rank['pushes_towards']=np.where(rank['mean_signed_shap']>0,POS_CLASS,NEG_CLASS)
    rank.to_csv(RESULTS_DIR/('shap_ranking_%s.csv'%shap_key[0]),index=False,encoding='utf-8-sig')
    print('\nTop 15 features:',shap_label)
    print(rank.head(15).to_string(index=False,float_format='%.4f'))
    if shap_key[0]=='audio_only':
        sv_audio=sv
        feat_cols_audio=list(X_shap.columns)
        xgb_bundle=bundle

### VIS 10 ROC curves

In [ ]:
best_feat='fused'


curves=[]
for model_name in BASE_MODELS:
    bundle=tuned_models.get((best_feat,model_name))
    if bundle is None:
        continue
    try:
        y_prob = bundle['fitted'].predict_proba(bundle['X_test'])[:, 1]
        fpr, tpr, _ = roc_curve(bundle['y_test'], y_prob)
        curves.append((model_name, fpr, tpr, roc_auc_score(bundle['y_test'], y_prob)))
    except Exception:
        continue
curves.sort(key=lambda t: -t[3])

SERIES = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#56B4E9']
ZX, ZY = 0.15, 0.85          # zoom window

fig, (ax, axz) = plt.subplots(1, 2, figsize=(12, 5.2),
                              gridspec_kw={'width_ratios': [1, 1], 'wspace': 0.22})

# ---- left: full range ----
for (m, fpr, tpr, auc), c in zip(curves, SERIES):
    ax.plot(fpr, tpr, color=c, lw=2, label='%s  (AUC = %.3f)' % (m, auc))
ax.plot([0, 1], [0, 1], ls='--', color=PALETTE['neutral'], lw=1,
        label='Random  (0.500)')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('Full range', fontweight='bold', loc='left', fontsize=11)
ax.legend(loc='lower right', frameon=False, fontsize=8.5)
ax.grid(alpha=0.45); ax.set_axisbelow(True)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
ax.add_patch(plt.Rectangle((0, ZY), ZX, 1.0 - ZY, fill=False,
                           ec=PALETTE['neutral'], lw=1.2, ls='--'))

# ---- right: zoomed ----
for (m, fpr, tpr, auc), c in zip(curves, SERIES):
    axz.plot(fpr, tpr, color=c, lw=2.2)
axz.set_xlim(0, ZX); axz.set_ylim(ZY, 1.005)
axz.set_xlabel('False positive rate'); axz.set_ylabel('True positive rate')
axz.set_title('Detail top-left corner (dashed box, left)',
              fontweight='bold', loc='left', fontsize=11)
axz.grid(alpha=0.45); axz.set_axisbelow(True)
for s in ['top', 'right']:
    axz.spines[s].set_visible(False)

spread = max(c[3] for c in curves) - min(c[3] for c in curves)
fig.suptitle('ROC curves by model %s   (all AUC within %.3f)'
             % (best_feat.replace('_', ' '), spread),
             fontweight='bold', fontsize=12, x=0.09, ha='left', y=1.02)

save_fig(fig, 'vis10_roc_curves')
plt.show()

In [ ]:
# Recreate model_order from tuned_models
model_order = list(dict.fromkeys(model for _, model in tuned_models.keys()))

print("Models found:")
print(model_order)

## 10. Save best model for demo app

In [ ]:
import joblib,json,sklearn,sys

MODEL_DIR=OUTPUT_DIR/'saved_models'
MODEL_DIR.mkdir(parents=True,exist_ok=True)

# Model selection uses the VALIDATION set, never the test set.
# The test set is touched exactly once, at the very end, for final reporting.
selected_val=val_df.loc[val_df.groupby('feature_set')['f1_macro'].idxmax()].copy()
overall=selected_val.loc[selected_val['f1_macro'].idxmax()]
saved=[]

for _,val_row in selected_val.iterrows():
    fs=val_row['feature_set']
    model_name=val_row['model']
    bundle=tuned_models.get((fs,model_name))
    if bundle is None:
        continue
    test_row=test_df[(test_df['feature_set']==fs)&(test_df['model']==model_name)].iloc[0]
    test_f1=f1_score(bundle['y_test'],bundle['fitted'].predict(bundle['X_test']),average='macro')
    assert np.isclose(test_f1,float(test_row['f1_macro']),atol=5e-5)
    joblib.dump(bundle['fitted'],MODEL_DIR/f'model_{fs}.joblib',compress=3)
    with open(MODEL_DIR/f'features_{fs}.json','w',encoding='utf-8') as f:
        json.dump(bundle['feat_cols'],f,indent=2)
    saved.append({
        'feature_set':fs,
        'model':model_name,
        'n_features':len(bundle['feat_cols']),
        'validation_f1':float(val_row['f1_macro']),
        'test_f1':float(test_row['f1_macro']),
        'test_auroc':float(test_row['auroc']),
        'test_accuracy':float(test_row['accuracy']),
        'is_overall_best':bool(fs==overall['feature_set'] and model_name==overall['model'])
    })
    print('Saved %-20s %-20s Val F1=%.4f Test F1=%.4f'%(fs,model_name,val_row['f1_macro'],test_f1))

joblib.dump(le,MODEL_DIR/'label_encoder.joblib')
with open(MODEL_DIR/'models_meta.json','w',encoding='utf-8') as f:
    json.dump({
        'models':saved,
        'classes':list(le.classes_),
        'selection_metric':'validation macro-F1',
        'sklearn_version':sklearn.__version__,
        'python_version':sys.version.split()[0]
    },f,indent=2)

print('\nSaved to:',MODEL_DIR)

## 11. Permutation importance (model-agnostic check)

SHAP is exact for trees but cannot explain SVM. Permutation importance shuffles each feature on the test set and measures the F1 drop. If SHAP (XGB) and permutation (SVM) agree on top features, the explanation is a property of the data, not the model.



In [ ]:
# Permutation importance runs on the SVM, not XGBoost, so it is model-agnostic
# with respect to tree structure. Compare its top-10 with the SHAP top-10:
#   shared features are likely data-level signals;
#   model-specific features should not be reported as universal AI defects.
SHAP_FS='audio_only'
svm_key=(SHAP_FS,'SVM (RBF)')
if svm_key not in tuned_models:
    raise KeyError('Audio-only SVM model not found.')
svm_bundle=tuned_models[svm_key]
print('Permutation importance: %s / %s (30 repeats)'%svm_key)

perm=permutation_importance(
    svm_bundle['fitted'],svm_bundle['X_test'],svm_bundle['y_test'],
    n_repeats=30,random_state=RANDOM_SEED,scoring='f1_macro',n_jobs=-1  # 30 shuffles per feature for stable estimates
)

perm_df=pd.DataFrame({
    'feature':svm_bundle['feat_cols'],
    'importance':perm.importances_mean,
    'std':perm.importances_std
}).sort_values('importance',ascending=False).reset_index(drop=True)

perm_df.to_csv(RESULTS_DIR/'svm_permutation_importance.csv',index=False)
print(perm_df.head(15).to_string(index=False,float_format='%.5f'))

top=perm_df.head(15).iloc[::-1]
fig,ax=plt.subplots(figsize=(8,6))
ax.barh(top['feature'],top['importance'],xerr=top['std'],
        color=PALETTE['human'],edgecolor='white',linewidth=0.6,
        error_kw={'ecolor':PALETTE['neutral'],'lw':0.8,'capsize':2})
ax.set_xlabel('Drop in macro-F1 when feature is shuffled')
ax.set_title('Permutation Importance: Audio-Only SVM')
tidy(ax,ygrid=False)
plt.tight_layout()
save_fig(fig,'vis9b_svm_permutation_importance')
plt.show()

if 'sv_audio' in globals() and 'feat_cols_audio' in globals():
    shap_imp=pd.DataFrame({
        'feature':feat_cols_audio,
        'shap_mean_abs':np.abs(sv_audio).mean(axis=0)
    })
    both=shap_imp.merge(perm_df[['feature','importance']],on='feature',how='inner')
    both['shap_rank']=both['shap_mean_abs'].rank(ascending=False)
    both['perm_rank']=both['importance'].rank(ascending=False)
    rho,pval=stats.spearmanr(both['shap_rank'],both['perm_rank'])
    top10_shap=set(both.nlargest(10,'shap_mean_abs')['feature'])
    top10_perm=set(both.nlargest(10,'importance')['feature'])
    overlap=top10_shap&top10_perm
    print('\nCommon features:',len(both))
    print('Spearman correlation: rho=%+.3f, p=%.3g'%(rho,pval))
    print('Top-10 overlap: %d/10'%len(overlap))
    print('Shared features:',', '.join(sorted(overlap)) or 'None')
    both.sort_values('shap_rank').to_csv(
        RESULTS_DIR/'shap_vs_permutation_agreement.csv',index=False
    )
else:
    print('Run the audio-only SHAP cell before comparing rankings.')

## 12. Per-track explainable quality report

Maps the top SHAP features for each track to a brief description of what they measure acoustically.


In [ ]:
# Plain-language label for a feature that deviates from the Human reference.
# NOTE: this helper was missing from the saved notebook (it was used below but
# never defined). Reconstructed to produce the same labels as the saved output,
# e.g. "Lower MFCC value than the Human reference".

def issue_label(feature, value, reference):
    direction = 'Higher' if value > reference else 'Lower'
    name = feature.lower()
    if 'mfcc' in name:
        kind = 'MFCC value '
    elif 'chroma' in name:
        kind = 'chroma value '
    elif 'flux' in name or 'onset' in name:
        kind = 'spectral change/activity '
    else:
        kind = ''
    return f'{direction} {kind}than the Human reference'


In [ ]:
# For each test track, the top SHAP features show which audio properties
# pushed the prediction toward AI or Human. A positive SHAP value means
# the feature increased the probability of the positive class (AI).
best_val=results_df[(results_df['split']=='validation')&(results_df['feature_set']=='fused')]
best_row=best_val.loc[best_val['f1_macro'].idxmax()]
best_key=('fused',best_row['model'])
bundle=tuned_models[best_key]
model=bundle['fitted']
X_train=bundle['X_train']
X_test=bundle['X_test']
audio_cols=[c for c in AUDIO_FEATS if c in X_test.columns]

imputer=model.named_steps['imputer']
X_train_imp=pd.DataFrame(imputer.transform(X_train),index=X_train.index,columns=bundle['feat_cols'])
X_test_imp=pd.DataFrame(imputer.transform(X_test),index=X_test.index,columns=bundle['feat_cols'])

human_mask=merged.loc[X_train.index,'source'].eq('Human').values
human_data=X_train_imp.loc[human_mask,audio_cols]
human_reference=human_data.median().fillna(X_train_imp[audio_cols].median())
human_mad=(human_data-human_reference).abs().median()

y_pred=model.predict(X_test)
y_prob=model.predict_proba(X_test)
base_prob=y_prob[np.arange(len(y_pred)),y_pred]

impacts=np.zeros((len(X_test),len(audio_cols)))
for j,feature in enumerate(audio_cols):
    X_changed=X_test.copy()
    X_changed[feature]=human_reference[feature]
    changed_prob=model.predict_proba(X_changed)
    impacts[:,j]=base_prob-changed_prob[np.arange(len(y_pred)),y_pred]

test_meta=merged.loc[X_test.index].reset_index(drop=True)
predicted=le.inverse_transform(y_pred)
rows=[]

for i,song in test_meta.iterrows():
    order=np.argsort(np.abs(impacts[i]))[::-1][:3]
    rec={
        'song_id':song.get('song_id',''),
        'title':song.get('title',''),
        'language':song.get('language',''),
        'true_source':song.get('source',''),
        'predicted_source':predicted[i],
        'correct':predicted[i]==song.get('source','')
    }
    for n,j in enumerate(order,1):
        feature=audio_cols[j]
        value=float(X_test_imp.iloc[i][feature])
        reference=float(human_reference[feature])
        mad=float(human_mad.get(feature,np.nan))
        robust_z=0.6745*(value-reference)/(mad+1e-9)
        issue=issue_label(feature,value,reference)
        if abs(robust_z)<1:
            issue='No strong deviation from the Human reference'
        rec[f'driver_{n}']=feature
        rec[f'value_{n}']=value
        rec[f'human_reference_{n}']=reference
        rec[f'robust_z_{n}']=robust_z
        rec[f'model_impact_{n}']=float(impacts[i,j])
        rec[f'possible_issue_{n}']=issue
    rows.append(rec)

quality_report=pd.DataFrame(rows)
quality_path=RESULTS_DIR/'technical_issue_report.csv'
quality_report.to_csv(quality_path,index=False,encoding='utf-8-sig')

print('Model:',best_row['model'])
print('Selected using validation F1: %.4f'%best_row['f1_macro'])
print('Technical issue report:',len(quality_report),'tracks')
print('Best fused model accuracy: %.4f'%quality_report['correct'].mean())
print('Saved:',quality_path)
display(quality_report.head(8))

## 13. Temporal acoustic analysis (RQ4)

Fits a linear slope to each acoustic feature across normalised track duration (t in [0, 1]) for a balanced sample of AI and human tracks. Mann-Whitney U tests with Benjamini-Hochberg correction are used to compare slope distributions between the two groups.


In [ ]:
# Temporal slopes use normalised duration (t in [0,1]) rather than absolute
# seconds so that slopes are directly comparable between tracks of different
# lengths. A slope of +100 Hz means the feature rises 100 Hz across the full
# track regardless of whether it is 2 or 6 minutes long.
# Implementation: t = np.linspace(0,1,len(values)); slope = np.polyfit(t,values,1)[0]
import librosa
from tqdm.auto import tqdm

SEGMENT_SR=22050
N_FFT,HOP=2048,512
EDGE_SKIP_SEC=5  # seconds to skip at start and end of each track

audio_df_all=pd.read_csv(AUDIO_CSV,low_memory=False)
ok=audio_df_all['ok'].astype(str).str.lower().eq('true')
seg_manifest=audio_df_all.loc[ok,['song_id','source','language','audio_path']].copy()
seg_manifest=seg_manifest.dropna(subset=['audio_path']).drop_duplicates('song_id')

parts=[]
for _,group in seg_manifest.groupby(['language','source']):
    parts.append(group.sample(n=min(50,len(group)),random_state=RANDOM_SEED))
seg_sample=pd.concat(parts,ignore_index=True)

def extract_temporal_trend(y,sr):
    S=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=HOP))
    mel=librosa.feature.melspectrogram(S=S**2,sr=sr)
    series={
        'rms_db':20*np.log10(librosa.feature.rms(S=S)[0]+1e-9),
        'spectral_centroid':librosa.feature.spectral_centroid(S=S,sr=sr)[0],
        'spectral_flatness':librosa.feature.spectral_flatness(S=S)[0],
        'spectral_rolloff':librosa.feature.spectral_rolloff(S=S,sr=sr)[0],
        'zcr':librosa.feature.zero_crossing_rate(y,frame_length=N_FFT,hop_length=HOP)[0],
        'onset_strength':librosa.onset.onset_strength(S=librosa.power_to_db(mel,ref=np.max),sr=sr,hop_length=HOP)
    }
    out={}
    for name,values in series.items():
        values=np.asarray(values,dtype=float)
        values=values[np.isfinite(values)]
        if len(values)<20:
            for suffix in ['_slope','_delta','_mean','_cv']:
                out[name+suffix]=np.nan
            continue
        t=np.linspace(0,1,len(values))  # normalised time axis [0,1] makes slopes comparable across track lengths
        q=max(1,len(values)//4)
        out[name+'_slope']=round(float(np.polyfit(t,values,1)[0]),5)
        out[name+'_delta']=round(float(values[-q:].mean()-values[:q].mean()),5)
        out[name+'_mean']=round(float(values.mean()),5)
        out[name+'_cv']=round(float(values.std()/(abs(values.mean())+1e-9)),5)
    return out

trend_records=[]
for _,row in tqdm(seg_sample.iterrows(),total=len(seg_sample),desc='Acoustic trend'):
    try:
        y,sr=librosa.load(str(row['audio_path']),sr=SEGMENT_SR,mono=True)
        edge=int(EDGE_SKIP_SEC*sr)
        if len(y)<(2*edge+10*sr):
            continue
        y=y[edge:-edge]
        rec=extract_temporal_trend(y,sr)
        rec.update({
            'song_id':row['song_id'],
            'source':row['source'],
            'language':row['language'],
            'duration_analysed_sec':round(len(y)/sr,1)
        })
        trend_records.append(rec)
    except Exception:
        continue

trend_df=pd.DataFrame(trend_records)
trend_path=RESULTS_DIR/'temporal_acoustic_trends.csv'
trend_df.to_csv(trend_path,index=False,encoding='utf-8-sig')

print('Tracks analysed:',len(trend_df))
print(trend_df.groupby(['language','source']).size().rename('n').to_string())
print('Saved:',trend_path)

# RQ4: temporal acoustic slope comparison


In [ ]:
# trend_df contains one row per track with a linear slope for each acoustic
# feature across normalised track duration. Positive slope = feature rises
# from start to end; negative slope = feature falls.
slope_cols=[c for c in trend_df.columns if c.endswith('_slope')]
rows=[]
for col in slope_cols:
    ai=pd.to_numeric(trend_df.loc[trend_df['source']=='AI',col],errors='coerce').dropna()
    hu=pd.to_numeric(trend_df.loc[trend_df['source']=='Human',col],errors='coerce').dropna()
    if len(ai)<20 or len(hu)<20:continue
    u,p=stats.mannwhitneyu(ai,hu,alternative='two-sided')
    effect=(2*u/(len(ai)*len(hu)))-1
    rows.append({'feature':col.replace('_slope',''),'AI_mean_slope':ai.mean(),'Human_mean_slope':hu.mean(),'AI_median_slope':ai.median(),'Human_median_slope':hu.median(),'rank_biserial':effect,'p_value':p,'n_AI':len(ai),'n_Human':len(hu)})
rq4=pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)
if not rq4.empty:
    p=rq4['p_value'].to_numpy()
    order=np.argsort(p)
    ranked=p[order]
    adjusted=ranked*len(p)/np.arange(1,len(p)+1)
    adjusted=np.minimum.accumulate(adjusted[::-1])[::-1]
    p_adjusted=np.empty_like(adjusted)
    p_adjusted[order]=np.clip(adjusted,0,1)
    rq4['p_adjusted']=p_adjusted
    rq4['significant']=rq4['p_adjusted']<0.05
    numeric=['AI_mean_slope','Human_mean_slope','AI_median_slope','Human_median_slope','rank_biserial','p_value','p_adjusted']
    rq4[numeric]=rq4[numeric].round(5)
rq4_path=RESULTS_DIR/'rq4_acoustic_trends.csv'
rq4.to_csv(rq4_path,index=False,encoding='utf-8-sig')
display(rq4)
print('Significant after correction: %d / %d'%(rq4['significant'].sum(),len(rq4)))
print('Positive effect: AI generally has a higher slope.')
print('Negative effect: AI generally has a lower slope.')
print('These are acoustic changes, not direct evidence of quality degradation.')
print('Saved:',rq4_path)

### VIS 11 Temporal feature evolution

In [ ]:
# VIS 11: Acoustic Feature Distributions (AI vs Human) 
import matplotlib.pyplot as plt
import numpy as np

try:
    from scipy.stats import gaussian_kde
    HAVE_KDE = True
except ImportError:
    HAVE_KDE = False

plot_features = [
    "rms_db",
    "spectral_centroid_mean",
    "spectral_rolloff_mean",
    "spectral_flatness_mean",
    "zcr_mean",
    "onset_strength_mean",
]

# Keep only features that exist
plot_features = [f for f in plot_features if f in audio_df.columns]

if len(plot_features) == 0:
    raise RuntimeError("None of the requested features exist in audio_df.")

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 8),
)

axes = axes.ravel()

COLORS = {
    "Human": "#0072B2",
    "AI": "#D55E00",
}

for ax, feat in zip(axes, plot_features):

    for src in ["Human", "AI"]:

        values = (
            audio_df.loc[audio_df["source"] == src, feat]
            .dropna()
            .astype(float)
            .values
        )

        if len(values) == 0:
            continue

        # Histogram
        ax.hist(
            values,
            bins=30,
            density=True,
            alpha=0.35,
            color=COLORS[src],
            edgecolor="white",
            linewidth=0.6,
            label=src,
        )

        # KDE curve
        if HAVE_KDE and len(values) > 5:
            kde = gaussian_kde(values)
            x = np.linspace(values.min(), values.max(), 300)
            ax.plot(
                x,
                kde(x),
                color=COLORS[src],
                linewidth=2.5,
            )

    ax.set_title(
        feat.replace("_", " ").replace("mean", "").title(),
        fontsize=11,
        fontweight="bold",
    )

    ax.grid(alpha=0.25)
    ax.set_ylabel("Density")
    ax.legend(frameon=False)

for i in range(len(plot_features), len(axes)):
    axes[i].axis("off")

fig.suptitle(
    "Overall Distribution of Key Acoustic Features by Source",
    fontsize=15,
    fontweight="bold",
)

plt.tight_layout()

save_fig(
    fig,
    "vis11_feature_distributions",
)

plt.show()

## 14. Cross-language generalisation (RQ1 extension)

Each fold trains on two languages and evaluates on the entirely unseen third, with full GridSearchCV repeated inside each fold. This tests whether the learned features transfer across languages or are language-specific.


In [ ]:
# Full GridSearchCV is repeated inside each held-out fold so hyperparameters
# are selected only on the two training languages. Using the in-distribution
# best settings for a held-out fold would be information leakage.
from sklearn.base import clone

y_all=le.transform(merged['source'].astype(str).to_numpy())
lang_all=merged['language'].astype(str).to_numpy()
assert len(y_all)==len(merged)==len(lang_all)

languages=[lang for lang in ['English','Hindi','Chinese'] if lang in set(lang_all)]  # only iterate languages present in the data
CL_SETS=[
    ('audio_only',AUDIO_FEATS),
    ('lyrics_only',LYRICS_COMBINED_FEATS),
    ('fused',FUSED_FEATS)
]

all_cross=[]
for feat_label,feat_cols in CL_SETS:
    print('\n=== Cross-language: %s ==='%feat_label)
    X_all=merged[feat_cols].apply(pd.to_numeric,errors='coerce').reset_index(drop=True)
    for held in languages:
        train_idx=np.where(lang_all!=held)[0]
        test_idx=np.where(lang_all==held)[0]
        X_train=X_all.iloc[train_idx]
        X_test=X_all.iloc[test_idx]
        y_train=y_all[train_idx]
        y_test=y_all[test_idx]
        cv=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=RANDOM_SEED)
        best_name=None
        best_cv=-np.inf
        best_model=None
        for name,base_model in BASE_MODELS.items():
            print('Tuning %-20s | held-out %s'%(name,held))
            search=GridSearchCV(
                clone(base_model),
                PARAM_GRIDS.get(name,{}),
                scoring='f1_macro',
                cv=cv,
                n_jobs=-1,
                refit=True,
                verbose=0,
                error_score='raise'
            )
            search.fit(X_train,y_train)
            if search.best_score_>best_cv:
                best_cv=search.best_score_
                best_name=name
                best_model=search.best_estimator_
        pred=best_model.predict(X_test)
        f1=f1_score(y_test,pred,average='macro',zero_division=0)
        acc=accuracy_score(y_test,pred)
        all_cross.append({
            'feature_set':feat_label,
            'held_out_language':held,
            'selected_model':best_name,
            'selection_cv_f1':best_cv,
            'train_n':len(train_idx),
            'test_n':len(test_idx),
            'macro_f1':f1,
            'accuracy':acc
        })
        print('Held-out %-8s | %-20s | CV F1=%.4f | Test F1=%.4f | Acc=%.4f'%(
            held,best_name,best_cv,f1,acc
        ))

cross_df=pd.DataFrame(all_cross)
metric_cols=['selection_cv_f1','macro_f1','accuracy']
cross_df[metric_cols]=cross_df[metric_cols].round(4)
cross_path=RESULTS_DIR/'cross_language_results.csv'
cross_df.to_csv(cross_path,index=False,encoding='utf-8-sig')

print('\n=== Cross-language results ===')
display(cross_df)

print('\n=== Mean cross-language performance ===')
cross_summary=cross_df.groupby('feature_set')[['macro_f1','accuracy']].agg(['mean','std']).round(4)
display(cross_summary)

print('\n=== Results by held-out language ===')
cross_pivot=cross_df.pivot(index='held_out_language',columns='feature_set',values='macro_f1').round(4)
display(cross_pivot)

print('Saved:',cross_path)

### VIS 12 Cross-language heatmap

In [ ]:
if not cross_df.empty:
    heat=cross_df.pivot(index='feature_set',columns='held_out_language',values='macro_f1')

    feat_order=[f for f in ['audio_only','lyrics_only','fused'] if f in heat.index]
    lang_order=[l for l in ['English','Hindi','Chinese'] if l in heat.columns]

    heat=heat.loc[feat_order,lang_order]
    # Rename for display only — the data column is still 'Chinese'.
    heat = heat.rename(columns={'Chinese': 'Mandarin'})

    fig,ax=plt.subplots(figsize=(7,4.5))
    sns.heatmap(
        heat,
        annot=True,
        fmt='.3f',
        cmap='YlGn',
        vmin=0.3,
        vmax=1.0,
        linewidths=0.5,
        ax=ax,
        annot_kws={'size':11,'weight':'bold'}
    )
    ax.set_title('Cross-Language Generalisation')
    ax.set_xlabel('Held-out language')
    ax.set_ylabel('Feature set')
    plt.tight_layout()
    save_fig(fig,'vis12_cross_language')
    plt.show()
else:
    print('cross_df is empty.')

## 15. Shuffled-label control

Trains on fused features with randomly permuted labels. Expected macro-F1 is approximately 0.50 on a balanced binary problem. A result substantially above 0.50 would indicate data leakage from the test set into the training pipeline.


In [ ]:
y = le.fit_transform(merged['source'].astype(str))
print(y.shape)

In [ ]:
# Shuffled-label control: labels are permuted before training and the full
# pipeline (imputation, scaling, GridSearchCV) is rerun on the shuffled data.
# Expected macro-F1 is ~0.50 on a balanced binary problem.
# Any result well above 0.50 would indicate data leakage.
from sklearn.base import clone

y_clean=le.transform(merged['source'].astype(str).to_numpy())
y_shuffled=np.random.RandomState(RANDOM_SEED).permutation(y_clean)  # fixed seed so results are reproducible
X_shuffled=merged[FUSED_FEATS].apply(pd.to_numeric,errors='coerce')

Xtr_s=X_shuffled.iloc[train_idx].copy()
Xte_s=X_shuffled.iloc[test_idx].copy()
ytr_s=y_shuffled[train_idx]
yte_s=y_shuffled[test_idx]

print('=== Shuffled-label control: Fused features ===')
print('Expected macro-F1: approximately 0.50\n')

shuffle_results=[]
for name,base_pipe in BASE_MODELS.items():
    model=clone(base_pipe)
    model.fit(Xtr_s,ytr_s)
    pred=model.predict(Xte_s)
    f1=f1_score(yte_s,pred,average='macro',zero_division=0)
    acc=accuracy_score(yte_s,pred)
    flag='CHECK' if f1>0.60 else 'OK'
    shuffle_results.append({'model':name,'f1_macro':f1,'accuracy':acc,'status':flag})
    print('%-22s F1=%.4f Acc=%.4f %s'%(name,f1,acc,flag))

shuffle_df=pd.DataFrame(shuffle_results)
shuffle_df[['f1_macro','accuracy']]=shuffle_df[['f1_macro','accuracy']].round(4)
shuffle_df.to_csv(RESULTS_DIR/'shuffled_label_control.csv',index=False,encoding='utf-8-sig')
print('\nSaved:',RESULTS_DIR/'shuffled_label_control.csv')

## 16. MuQ-Eval quality analysis (RQ4 and RQ5)

MuQ-Eval provides per-track quality scores (Musical Impression, 1 to 5). Each track is scored using five equally spaced 10-second windows, skipping the first and last five seconds of the track.


In [ ]:
# MuQ-Eval scores each track using five equally-spaced 10-second windows.
# The first and last 5 seconds are skipped to avoid intro silence and
# outro fade artefacts that would bias the quality estimate.
# The five window scores are averaged to give a single per-track MI score.
if not MUQ_CSV.exists():
    print('MuQ scores not found. Run MuQ scoring first.')
    muq_scores=pd.DataFrame()
else:
    muq_scores=pd.read_csv(MUQ_CSV,low_memory=False)
    ok=muq_scores['ok'].astype(str).str.lower().eq('true')
    muq_scores=muq_scores.loc[ok].drop_duplicates('song_id',keep='last').copy()
    muq_scores=muq_scores[muq_scores['source'].isin(['AI','Human'])]
    muq_scores['muq_mi_mean']=pd.to_numeric(muq_scores['muq_mi_mean'],errors='coerce')  # coerce handles any string 'nan' values

    print('MuQ tracks:',len(muq_scores))
    print(muq_scores.groupby(['language','source']).size().rename('n').to_string())

    print('\nEstimated quality-score summary:')
    print(muq_scores.groupby('source')['muq_mi_mean'].agg(['count','mean','std','median']).round(3).to_string())

    muq_ai=muq_scores.loc[muq_scores['source']=='AI','muq_mi_mean'].dropna()
    muq_hu=muq_scores.loc[muq_scores['source']=='Human','muq_mi_mean'].dropna()

    if len(muq_ai)>5 and len(muq_hu)>5:
        u,p=stats.mannwhitneyu(muq_ai,muq_hu,alternative='two-sided')
        effect=(2*u/(len(muq_ai)*len(muq_hu)))-1
        print('\nMann-Whitney U=%.0f | p=%.3g | rank-biserial=%+.3f'%(u,p,effect))

    rng=np.random.RandomState(RANDOM_SEED)

    def mean_ci(values,n_boot=2000):
        values=np.asarray(values,float)
        values=values[np.isfinite(values)]
        if len(values)<3:return np.nan,np.nan,np.nan
        boot=[np.mean(rng.choice(values,len(values),replace=True)) for _ in range(n_boot)]
        return values.mean(),np.percentile(boot,2.5),np.percentile(boot,97.5)

    groups=[('Overall',muq_scores)]

    langs=[l for l in ['English','Hindi','Chinese'] if l in set(muq_scores['language'].dropna())]
    lang_display={'Chinese':'Mandarin'}

    for lang in langs:
        groups.append((
            lang_display.get(lang,lang),
            muq_scores[muq_scores['language']==lang]
        ))

    # Exactly 5 genres.
    # Folk and Folk/Acoustic are combined into one Folk category.
    genre_groups={
        'Folk':['Folk','Folk/Acoustic'],
        'Hip Hop':['Hip-Hop/Rap'],
        'Pop':['Pop'],
        'R&B':['R&B/Soul'],
        'Rock':['Rock']
    }

    for display_name,source_names in genre_groups.items():
        mask=muq_scores['final_genre'].isin(source_names)
        if mask.any():
            groups.append((
                display_name,
                muq_scores[mask]
            ))

    fig,(axd,axp)=plt.subplots(
        1,2,
        figsize=(12,5.6),
        gridspec_kw={'width_ratios':[1,1.35],'wspace':0.28}
    )

    for src,color in CLASS_COLORS.items():
        values=muq_scores.loc[
            muq_scores['source']==src,
            'muq_mi_mean'
        ].dropna()

        axd.hist(
            values,
            bins=35,
            alpha=0.55,
            color=color,
            edgecolor='white',
            linewidth=0.5,
            label='%s (n=%d)'%(src,len(values))
        )

        axd.axvline(
            values.mean(),
            color=color,
            ls='--',
            lw=1.5
        )

    axd.set_xlim(1,5)
    axd.set_xlabel('MuQ-Eval Musical Impression score')
    axd.set_ylabel('Tracks')
    axd.set_title(
        'Estimated quality-score distribution',
        fontweight='semibold',
        loc='left'
    )
    axd.legend(frameon=False)
    axd.grid(axis='y',alpha=0.45)
    axd.grid(axis='x',visible=False)

    ypos=np.arange(len(groups))[::-1]

    for yi,(label,sub) in zip(ypos,groups):
        values={}

        for src,color in CLASS_COLORS.items():
            mean,low,high=mean_ci(
                sub.loc[
                    sub['source']==src,
                    'muq_mi_mean'
                ]
            )

            if np.isfinite(mean):
                values[src]=mean

                axp.plot(
                    [low,high],
                    [yi,yi],
                    color=color,
                    lw=2.6,
                    solid_capstyle='round',
                    alpha=0.85
                )

                axp.scatter(
                    mean,
                    yi,
                    s=70,
                    color=color,
                    edgecolor='white',
                    linewidth=1.1,
                    zorder=4
                )

        if {'AI','Human'}.issubset(values):
            axp.annotate(
                'Δ %+.2f'%(values['AI']-values['Human']),
                (max(values.values())+0.04,yi),
                va='center',
                fontsize=7,
                color=PALETTE['neutral']
            )

    axp.set_yticks(ypos)
    axp.set_yticklabels(
        [label for label,_ in groups],
        fontsize=8
    )
    axp.tick_params(axis='y',pad=0)
    axp.set_xlim(1,5)
    axp.set_xlabel('Mean MI score with 95% bootstrap CI')
    axp.set_title(
        'Estimated quality by language and genre',
        fontweight='semibold',
        loc='left'
    )

    handles=[
        plt.Line2D(
            [0],[0],
            marker='o',
            color=color,
            lw=2.6,
            markersize=7,
            markeredgecolor='white',
            label=src
        )
        for src,color in CLASS_COLORS.items()
    ]

    axp.legend(
        handles=handles,
        frameon=False,
        loc='lower right'
    )

    axp.grid(axis='x',alpha=0.45)
    axp.grid(axis='y',visible=False)

    for ax in [axd,axp]:
        for spine in ['top','right']:
            ax.spines[spine].set_visible(False)

    plt.tight_layout()
    save_fig(fig,'muq1_quality_by_class')
    plt.show()

### 16.1 Taxonomy vs quality correlations

In [ ]:
# Maps individual feature names to broader acoustic categories.
# Used to group SHAP attributions into interpretable labels
# in the per-track quality report.
TAXONOMY={
    'Loudness and clipping':['lufs_integrated','sample_peak_dbfs','clip_rate_per_sec'],
    'Dynamic range':['dynamic_range_db','crest_factor_db'],
    'Spectral balance':['band_energy_low','band_energy_mid','band_energy_high','spectral_centroid_mean','spectral_flatness_mean','spectral_contrast_mean'],
    'Transient character':['onset_strength_mean','onset_strength_std'],
    'Rhythm':['beat_regularity_std','bpm'],
    'Stereo image':['stereo_lr_ratio','stereo_width'],
    'Lyric repetition':['longest_repeat_ratio','bigram_rep_ratio','ttr']
}
CONFOUND_SENSITIVE={'lufs_integrated','sample_peak_dbfs','clip_rate_per_sec'}

def bh_adjust(values):
    values=np.asarray(values,float)
    out=np.full(len(values),np.nan)
    valid=np.isfinite(values)
    p=values[valid]
    if not len(p):return out
    order=np.argsort(p)
    ranked=p[order]
    adjusted=ranked*len(p)/np.arange(1,len(p)+1)
    adjusted=np.minimum.accumulate(adjusted[::-1])[::-1]
    temp=np.empty(len(p))
    temp[order]=np.clip(adjusted,0,1)
    out[valid]=temp
    return out

def spearman_result(data,feature):
    pair=data[[feature,'muq_mi_mean']].apply(pd.to_numeric,errors='coerce').dropna()
    if len(pair)<30 or pair[feature].nunique()<2:
        return np.nan,np.nan,len(pair)
    r,p=stats.spearmanr(pair[feature],pair['muq_mi_mean'])
    return r,p,len(pair)

if muq_scores.empty:
    print('MuQ scores are unavailable.')
else:
    audio_tax=audio_df.copy()
    if {'hard_clip_count','duration_seconds'}.issubset(audio_tax.columns):
        audio_tax['clip_rate_per_sec']=pd.to_numeric(audio_tax['hard_clip_count'],errors='coerce')/pd.to_numeric(audio_tax['duration_seconds'],errors='coerce').clip(lower=1e-6)

    features=list(dict.fromkeys(f for group in TAXONOMY.values() for f in group))
    audio_cols=['song_id']+[f for f in features if f in audio_tax.columns]
    lyric_cols=['song_id']+[f for f in features if f in lyrics_df.columns and f not in audio_cols]

    muq_tax=muq_scores[['song_id','source','language','final_genre','muq_mi_mean']].copy()
    muq_tax=muq_tax.merge(audio_tax[audio_cols].drop_duplicates('song_id',keep='last'),on='song_id',how='left')
    if len(lyric_cols)>1:
        muq_tax=muq_tax.merge(lyrics_df[lyric_cols].drop_duplicates('song_id',keep='last'),on='song_id',how='left')

    rows=[]
    for category,category_features in TAXONOMY.items():
        for feature in category_features:
            if feature not in muq_tax.columns:continue
            r_all,p_all,n_all=spearman_result(muq_tax,feature)
            r_ai,p_ai,n_ai=spearman_result(muq_tax[muq_tax['source']=='AI'],feature)
            r_hu,p_hu,n_hu=spearman_result(muq_tax[muq_tax['source']=='Human'],feature)
            rows.append({
                'category':category,'feature':feature,
                'confound_sensitive':feature in CONFOUND_SENSITIVE,
                'r_pooled':r_all,'p_pooled':p_all,'n_pooled':n_all,
                'r_AI':r_ai,'p_AI':p_ai,'n_AI':n_ai,
                'r_Human':r_hu,'p_Human':p_hu,'n_Human':n_hu
            })

    tax_df=pd.DataFrame(rows)
    if tax_df.empty:
        print('No taxonomy features were available.')
    else:
        tax_df['q_pooled']=bh_adjust(tax_df['p_pooled'])
        tax_df['q_AI']=bh_adjust(tax_df['p_AI'])
        tax_df['q_Human']=bh_adjust(tax_df['p_Human'])
        tax_df['within_max']=tax_df[['r_AI','r_Human']].abs().max(axis=1)
        tax_df=tax_df.sort_values('within_max',ascending=False).reset_index(drop=True)
        tax_path=RESULTS_DIR/'taxonomy_quality_correlations.csv'
        tax_df.drop(columns='within_max').to_csv(tax_path,index=False,encoding='utf-8-sig')
        display(tax_df[['category','feature','confound_sensitive','r_pooled','r_AI','q_AI','r_Human','q_Human']].round(4))

        plot_df=tax_df.dropna(subset=['within_max']).head(16).iloc[::-1]
        categories=list(TAXONOMY)
        palette={c:p for c,p in zip(categories,sns.color_palette('tab10',len(categories)))}
        colours=[palette[c] for c in plot_df['category']]
        y_pos=np.arange(len(plot_df))

        fig,ax=plt.subplots(figsize=(10,7.5))
        ax.barh(y_pos,plot_df['r_pooled'],color=colours,alpha=0.35,edgecolor='white',height=0.62,label='Pooled')
        ax.scatter(plot_df['r_AI'],y_pos,marker='o',s=52,color=CLASS_COLORS['AI'],edgecolor='white',linewidth=0.8,zorder=4,label='Within AI')
        ax.scatter(plot_df['r_Human'],y_pos,marker='D',s=48,color=CLASS_COLORS['Human'],edgecolor='white',linewidth=0.8,zorder=4,label='Within Human')
        ax.axvline(0,color='black',lw=1)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(plot_df['feature'])
        ax.set_xlabel('Spearman correlation with MuQ-Eval MI score')
        ax.set_title('Candidate Technical Indicators and Estimated Quality',fontweight='semibold',loc='left')
        ax.legend(frameon=False,loc='lower right')
        ax.grid(axis='x',alpha=0.45)
        ax.grid(axis='y',visible=False)
        for spine in ['top','right','left']:
            ax.spines[spine].set_visible(False)
        ax.tick_params(axis='y',length=0)
        plt.tight_layout()
        save_fig(fig,'muq2_taxonomy_correlations')
        plt.show()
        print('Saved:',tax_path)

### 16.2 RQ4 Temporal quality trajectory

In [ ]:
if not muq_scores.empty and 'muq_mi_slope' in muq_scores.columns:
    muq_scores['muq_mi_slope']=pd.to_numeric(muq_scores['muq_mi_slope'],errors='coerce')
    print('MI slope by source (negative = estimated score falls toward the end):')  # negative slope = quality estimated to drop toward track end
    print(muq_scores.groupby('source')['muq_mi_slope'].agg(['count','mean','std','median']).round(4).to_string())
    s_ai=muq_scores.loc[muq_scores['source']=='AI','muq_mi_slope'].dropna()
    s_hu=muq_scores.loc[muq_scores['source']=='Human','muq_mi_slope'].dropna()
    if len(s_ai)>5 and len(s_hu)>5:
        u,p=stats.mannwhitneyu(s_ai,s_hu,alternative='two-sided')
        effect=(2*u/(len(s_ai)*len(s_hu)))-1
        print('\nMann-Whitney U=%.0f | p=%.3g | rank-biserial=%+.3f'%(u,p,effect))
        print('AI negative slope: %.1f%%'%(100*(s_ai<0).mean()))
        print('Human negative slope: %.1f%%'%(100*(s_hu<0).mean()))
    fig,axes=plt.subplots(1,2,figsize=(13,4.4))
    for src,color in CLASS_COLORS.items():
        values=muq_scores.loc[muq_scores['source']==src,'muq_mi_slope'].dropna()
        axes[0].hist(values,bins=40,alpha=0.55,label='%s (n=%d)'%(src,len(values)),color=color,edgecolor='white')
    axes[0].axvline(0,color='black',ls='--',lw=1.2)
    axes[0].set_xlabel('MI slope across normalized track position')
    axes[0].set_ylabel('Tracks')
    axes[0].set_title('Distribution of MuQ-Eval Score Trends',fontweight='semibold')
    axes[0].legend(frameon=False)
    axes[0].grid(axis='y',alpha=0.4);axes[0].grid(axis='x',visible=False)
    for src,color in CLASS_COLORS.items():
        arrays=[]
        for value in muq_scores.loc[muq_scores['source']==src,'muq_mi_windows'].dropna():
            try:
                scores=[float(x) for x in str(value).split(';') if x.strip()]
                if len(scores)==5:arrays.append(scores)
            except Exception:continue
        if arrays:
            matrix=np.asarray(arrays,float)
            mean=matrix.mean(axis=0)
            ci=1.96*matrix.std(axis=0,ddof=1)/np.sqrt(len(matrix))
            position=np.linspace(0,100,matrix.shape[1])
            axes[1].plot(position,mean,'o-',color=color,label='%s (n=%d)'%(src,len(matrix)),lw=2)
            axes[1].fill_between(position,mean-ci,mean+ci,color=color,alpha=0.18)
    axes[1].set_xlabel('Normalized track position (%)')
    axes[1].set_ylabel('Mean MuQ-Eval MI score')
    axes[1].set_title('Mean Estimated Quality Trajectory',fontweight='semibold')
    axes[1].legend(frameon=False)
    axes[1].grid(alpha=0.4)
    for ax in axes:
        for spine in ['top','right']:ax.spines[spine].set_visible(False)
    plt.tight_layout()
    save_fig(fig,'muq3_temporal_quality_trend')
    plt.show()
else:
    print('MuQ slope scores are unavailable.')

##  Final summary

In [ ]:
print('=' * 60)
print('HEADLINE RESULTS — AI Music Technical Quality Detector')
print('=' * 60)
print('Best model  : %s | %s' % (best_row['model'], best_row['feature_set']))
# best_row comes from validation-set idxmax, so f1_macro is the validation F1
# used for model selection, not the held-out test F1.
print('Val F1 (selection criterion): %.4f' % best_row['f1_macro'])
# Look up the actual test result for this model and feature set
_test_match = test_df[(test_df['feature_set']==best_row['feature_set']) &
                      (test_df['model']==best_row['model'])]
_test_f1  = _test_match['f1_macro'].values[0] if len(_test_match) else float('nan')
_test_auc = _test_match['auroc'].values[0]    if len(_test_match) else float('nan')
print('Test F1     : %.4f' % _test_f1)
print('Test AUROC  : %.4f' % _test_auc)
print('Test Acc    : %.4f' % best_row['accuracy'])
print()
print('Dataset     : 1800 songs (300 x 6: EN/HI/ZH x AI/Human)')
print('Split       : 60/20/20 stratified train/val/test')
# LYRICS_FEATS holds only 15 handcrafted features; 384 embedding dimensions
# are in EMBEDDING_FEATS. The fused vector is 78 + 15 + 384 = 477.
print('Features    : %d audio + 15 handcrafted + 384 embeddings = %d fused' % (
    len(AUDIO_FEATS), len(FUSED_FEATS)))
print()
if not cross_df.empty:
    audio_cl = cross_df[cross_df['feature_set']=='audio_only']['macro_f1'].mean()
    print('Cross-lang F1 (audio, avg) : %.4f' % audio_cl)
print()
print('=' * 60)